In [0]:
import requests
import json
from pyspark.sql import SparkSession

#Definition des variables

In [0]:
CLIENT_ID = ""
CLIENT_SECRET = ""
USER_ACCESS_TYPE = "TOAST_MACHINE_CLIENT"
BASE_URL = "https://ws-api.toasttab.com"

RESTAURANT_GUID = ""


#recuperation du token

In [0]:
auth_url = f"{BASE_URL}/authentication/v1/authentication/login"

auth_payload = {
    "clientId": CLIENT_ID,
    "clientSecret": CLIENT_SECRET,
    "userAccessType": USER_ACCESS_TYPE
}

auth_headers = {
    "Content-Type": "application/json"
}

auth_response = requests.post(auth_url, headers=auth_headers, json=auth_payload)

print("Status auth :", auth_response.status_code)
print(auth_response.text)

In [0]:
auth_data = auth_response.json()
access_token = auth_data["token"]["accessToken"]

print("Token récupéré avec succès")

#recuperation des id de restaurant

In [0]:
restaurant_url = f"{BASE_URL}/restaurants/v1/restaurants/{RESTAURANT_GUID}"

restaurant_headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
    "Toast-Restaurant-External-ID": RESTAURANT_GUID
}

restaurant_response = requests.get(restaurant_url, headers=restaurant_headers)

print("Status restaurant :", restaurant_response.status_code)
print(restaurant_response.text)

#transformer la reponse en JSON

In [0]:
restaurant_data = restaurant_response.json()

print(type(restaurant_data))
print(restaurant_data.keys())

#Charge les donnees dans spark dataframe

In [0]:
from pyspark.sql import functions as F

json_str = json.dumps(restaurant_data)

df_bronze = spark.createDataFrame(
    [(restaurant_data["guid"], json_str)],
    ["restaurant_guid", "raw_json"]
).withColumn("ingestion_timestamp", F.current_timestamp())

df_bronze.write.mode("overwrite").saveAsTable("project1.bronze.toast_restaurant_raw")

In [0]:
display(spark.table("project1.bronze.toast_restaurant_raw"))

In [0]:
%sql
select * from project1.bronze.toast_restaurant_raw
